# MVA Track 2 · NB08R2
## Audited approved-drug coverage with exact component-aware ChEMBL target mapping

**v1.0.2 precision patch**

This notebook supersedes NB08R2 v1.0.1.

v1.0.1 correctly fixed `NOT_RUN` vs `0`, approval filtering, salt/name deduplication,
and the proteasome positive control, but its free-text ChEMBL target search was too permissive.
For example, text searches for HSF1 or "26S proteasome" could return unrelated targets.

v1.0.2 therefore uses the frozen single-protein ChEMBL target ID only to obtain the exact
UniProt accession, and then calls the documented ChEMBL filter:

`target_components__accession=<UniProt accession>`

This systematically captures complexes/families that truly contain the protein component,
without fuzzy text matching.

This remains a **coverage and falsification audit**, not a drug-ranking engine.


In [ ]:
from pathlib import Path
import os, re, json, time, platform, shutil, zipfile
import pandas as pd, numpy as np, requests

# -----------------------------
# NB08R2 v1.0.2 public runtime bootstrap
# -----------------------------
# Priority: repository/local snapshots -> optional environment override -> public fallback.

CWD = Path.cwd()
RUNTIME = CWD / "MVA_NB08R2_runtime"
RUNTIME.mkdir(parents=True, exist_ok=True)

def valid_snapshot_dir(p: Path) -> bool:
    return ((p / "drugcentral_interactions_snapshot.csv").exists() and
            (p / "drugcentral_approved_snapshot_NORMALIZED.csv").exists())

snapshot_candidates = [CWD / "core" / "snapshots", CWD / "snapshots", CWD.parent / "snapshots"]
if os.environ.get("MVA_NB08R2_SNAPSHOTS"):
    snapshot_candidates.insert(0, Path(os.environ["MVA_NB08R2_SNAPSHOTS"]))
SNAP = next((q for q in snapshot_candidates if valid_snapshot_dir(q)), None)
SNAPSHOT_MODE = "FROZEN"
PERSIST = None
if SNAP is None:
    SNAPSHOT_MODE = "LIVE_FALLBACK"
    SNAP = RUNTIME / "snapshots"
    SNAP.mkdir(parents=True, exist_ok=True)
    import io, gzip
    rr = requests.get("https://unmtid-dbs.net/download/DrugCentral/2021_09_01/drug.target.interaction.tsv.gz", timeout=120); rr.raise_for_status()
    pd.read_csv(io.BytesIO(gzip.decompress(rr.content)), sep="\t", low_memory=False).to_csv(SNAP/"drugcentral_interactions_snapshot.csv", index=False)
    rr = requests.get("https://drugcentral.org/static/FDA%2BEMA%2BPMDA_Approved.csv", timeout=90); rr.raise_for_status()
    from io import StringIO
    app = pd.read_csv(StringIO(rr.content.decode("utf-8",errors="replace")), header=None, names=["STRUCT_ID","DRUG_NAME"])
    app["STRUCT_ID"] = pd.to_numeric(app["STRUCT_ID"], errors="coerce").astype("Int64")
    app["DRUG_NAME"] = app["DRUG_NAME"].astype(str)
    app.dropna(subset=["STRUCT_ID"]).drop_duplicates().to_csv(SNAP/"drugcentral_approved_snapshot_NORMALIZED.csv", index=False)
    print("WARNING: using LIVE_FALLBACK DrugCentral snapshots")
assert valid_snapshot_dir(SNAP), f"Snapshot bootstrap failed; resolved SNAP={SNAP}"
BASE = RUNTIME
OUT = BASE / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
print("Python:", platform.python_version())
print("pandas:", pd.__version__)
print("requests:", requests.__version__)
print("SNAPSHOT_MODE:", SNAPSHOT_MODE)
print("SNAP:", SNAP.resolve())
print("OUT:", OUT.resolve())
print("PERSIST:", str(PERSIST) if PERSIST else "LOCAL_ONLY")


## 1. Frozen mechanistic registry

Each branch has one **frozen core single-protein ChEMBL target ID** from the original NB08 target-resolution audit.

The core ID is not treated as the only representation. It is used to retrieve the protein's exact
UniProt accession. v1.0.2 then asks ChEMBL for **all human targets whose `target_components`
contain that accession**, which can include genuine protein complexes or families.

No free-text target search is used.


In [ ]:
REGISTRY = pd.DataFrame([
    ["ABUNDANCE_PROTEOSTASIS","HSF1","ACTIVATE","PROXIMAL","CHEMBL5869"],
    ["ABUNDANCE_PROTEOSTASIS","HSP90AA1","SUPPORT_NOT_INHIBIT","PROXIMAL","CHEMBL3880"],
    ["ABUNDANCE_PROTEOSTASIS","PSMB5","INHIBIT_PROBE_ONLY","PROBE_ONLY","CHEMBL4662"],
    ["ABUNDANCE_STABILITY_PTM","SIRT2","ACTIVATE","PROXIMAL","CHEMBL4462"],
    ["ABUNDANCE_STABILITY_PTM","CREBBP","INHIBIT_EXPLORATORY","EXPLORATORY","CHEMBL5747"],
    ["FUNCTION_KARD_PP2A","AURKB","INHIBIT_PARTIALLY_PROBE_ONLY","PROBE_ONLY","CHEMBL2185"],
    ["FUNCTION_KARD_PP2A","PPP2CA","ACTIVATE_LOCALIZED","PROXIMAL","CHEMBL4703"],
    ["FUNCTION_KARD_PP2A","PLK1","PRESERVE","ANTITARGET","CHEMBL3024"],
    ["FUNCTION_KARD_PP2A","CDK1","PRESERVE","ANTITARGET","CHEMBL308"],
    ["FUNCTION_SAC","TTK","PRESERVE","ANTITARGET","CHEMBL3983"],
    ["DOWNSTREAM_IFN","JAK1","INHIBIT_IF_ACTIVE","DOWNSTREAM","CHEMBL2835"],
    ["DOWNSTREAM_IFN","JAK2","INHIBIT_IF_ACTIVE","DOWNSTREAM","CHEMBL2971"],
    ["DOWNSTREAM_MTOR_AUTOPHAGY","MTOR","INHIBIT_CONDITIONALLY","DOWNSTREAM","CHEMBL2842"],
], columns=["branch","gene_symbol","desired_action","role","core_target_chembl_id"])
REGISTRY


## 2. Frozen DrugCentral snapshots

The historical approval file was headerless. NB08 v1.0.1 read it with the default CSV header and never actually used it to filter interactions.  
Here approval is explicit and auditable.


In [ ]:
inter_path = SNAP/"drugcentral_interactions_snapshot.csv"
approved_path = SNAP/"drugcentral_approved_snapshot_NORMALIZED.csv"

assert inter_path.exists(), f"Missing {inter_path}"
assert approved_path.exists(), f"Missing {approved_path}"

dc = pd.read_csv(inter_path, low_memory=False)
approved = pd.read_csv(approved_path)
approved["STRUCT_ID"] = pd.to_numeric(approved["STRUCT_ID"], errors="coerce").astype("Int64")
APPROVED_IDS = set(approved["STRUCT_ID"].dropna().astype(int))

dc["STRUCT_ID"] = pd.to_numeric(dc["STRUCT_ID"], errors="coerce").astype("Int64")
dc["approved_snapshot"] = dc["STRUCT_ID"].isin(APPROVED_IDS)
print("DrugCentral rows:", len(dc))
print("Approved structure IDs:", len(APPROVED_IDS))
print("Approved interaction rows:", int(dc["approved_snapshot"].sum()))


## 3. DrugCentral target audit

Counts are unique approved `STRUCT_ID`s. Raw names are retained only for readability.  
This prevents salts/name variants from inflating coverage.


In [ ]:
dc_rows=[]
for _, r in REGISTRY.iterrows():
    z = dc[
        dc["approved_snapshot"] &
        dc["GENE"].astype(str).str.upper().eq(r.gene_symbol.upper()) &
        dc["ORGANISM"].astype(str).str.lower().eq("homo sapiens")
    ].copy()
    ids = sorted(set(z["STRUCT_ID"].dropna().astype(int)))
    dc_rows.append({
        "branch":r.branch,
        "target_symbol":r.gene_symbol,
        "drugcentral_query_state":"HIT" if len(ids) else "NO_HIT",
        "drugcentral_unique_approved_structures":len(ids),
        "drugcentral_struct_ids":";".join(map(str,ids)),
        "drugcentral_drugs":";".join(sorted(set(z["DRUG_NAME"].dropna().astype(str).str.lower())))
    })
DC_AUDIT=pd.DataFrame(dc_rows)
display(DC_AUDIT)

# Internal positive-control audit: proteasome inhibitors must be recoverable.
for drug in ["bortezomib","carfilzomib","ixazomib"]:
    z=dc[(dc["approved_snapshot"]) & (dc["DRUG_NAME"].astype(str).str.lower().eq(drug))]
    print(drug, "PSMB5 rows:", int(z["GENE"].astype(str).eq("PSMB5").sum()),
          "proteasome-gene rows:", int(z["GENE"].astype(str).str.startswith("PSMB").sum()))


## 4. ChEMBL exact component-aware target-resolution helpers

ChEMBL officially supports filtering targets by a UniProt accession contained in
`target_components`. This is used instead of full-text search.

Query-state semantics remain unchanged:
- completed query with records: `HIT`;
- completed query with zero records: `NO_HIT`;
- skipped query: `NOT_RUN`;
- failed request: `ERROR`.


In [ ]:
CHEMBL="https://www.ebi.ac.uk/chembl/api/data"
SESSION=requests.Session()
SESSION.headers.update({"User-Agent":"MVA-Track2-NB08R2/1.0.2"})

def get_json(url, params=None, timeout=45, retries=3):
    err=None
    for k in range(retries):
        try:
            rr=SESSION.get(url, params=params, timeout=timeout)
            rr.raise_for_status()
            return rr.json(), None
        except Exception as e:
            err=repr(e)
            time.sleep(1.5*(k+1))
    return None, err

def get_target_detail(target_chembl_id):
    data,err=get_json(f"{CHEMBL}/target/{target_chembl_id}.json")
    if err:
        return None, "ERROR", err
    return data, "HIT", None

def exact_accessions_from_core_target(target_chembl_id):
    data,state,err=get_target_detail(target_chembl_id)
    if state=="ERROR":
        return [], state, err
    comps=data.get("target_components") or []
    accs=sorted({str(x.get("accession")) for x in comps if x.get("accession")})
    return accs, ("HIT" if accs else "NO_HIT"), None

def targets_containing_accession(accession):
    # Documented ChEMBL filter; no fuzzy text search.
    data,err=get_json(
        f"{CHEMBL}/target.json",
        {"target_components__accession":accession, "limit":1000}
    )
    if err:
        return [], "ERROR", err
    rows=[]
    for t in data.get("targets",[]):
        if str(t.get("organism") or "") != "Homo sapiens":
            continue
        typ=str(t.get("target_type") or "")
        if typ not in {"SINGLE PROTEIN","PROTEIN COMPLEX","PROTEIN FAMILY","PROTEIN COMPLEX GROUP"}:
            continue
        # Defensive verification: returned target must actually include accession.
        comps=t.get("target_components") or []
        accs={str(x.get("accession")) for x in comps if x.get("accession")}
        if accession not in accs:
            continue
        rows.append({
            "target_chembl_id":t.get("target_chembl_id"),
            "pref_name":t.get("pref_name"),
            "target_type":typ,
            "organism":t.get("organism"),
            "matched_accession":accession,
            "all_component_accessions":";".join(sorted(accs)),
        })
    return rows, ("HIT" if rows else "NO_HIT"), None

def molecule_info(mid, cache={}):
    if mid in cache:
        return cache[mid]
    d,e=get_json(f"{CHEMBL}/molecule/{mid}.json")
    if e:
        cache[mid]={"molecule_chembl_id":mid,"query_error":e}
    else:
        h=d.get("molecule_hierarchy") or {}
        cache[mid]={
            "molecule_chembl_id":mid,
            "pref_name":d.get("pref_name"),
            "max_phase":d.get("max_phase"),
            "first_approval":d.get("first_approval"),
            "parent_chembl_id":h.get("parent_chembl_id") or mid,
            "query_error":None,
        }
    return cache[mid]

def mechanisms_for_target(tid):
    data,err=get_json(f"{CHEMBL}/mechanism.json", {"target_chembl_id":tid,"limit":1000})
    if err:
        return [], "ERROR", err
    return data.get("mechanisms",[]), "OK", None


## 5. Run exact component-aware ChEMBL mechanism audit

For each frozen core target:

1. fetch its exact UniProt accession(s);
2. retrieve every human ChEMBL protein target that actually contains that accession;
3. query mechanisms for only those verified representations;
4. canonicalize molecules to their parent ChEMBL ID.

This captures real complexes containing the protein and excludes fuzzy name matches.


In [ ]:
target_rows=[]
mech_rows=[]
resolution_rows=[]

for _,r in REGISTRY.iterrows():
    accs,state,err=exact_accessions_from_core_target(r.core_target_chembl_id)
    resolution_rows.append({
        "gene_symbol":r.gene_symbol,
        "core_target_chembl_id":r.core_target_chembl_id,
        "core_accession_state":state,
        "core_accessions":";".join(accs),
        "error":err,
    })
    if state=="ERROR":
        continue
    if not accs:
        continue

    seen={}
    for acc in accs:
        hits,tstate,terr=targets_containing_accession(acc)
        if tstate=="ERROR":
            resolution_rows.append({
                "gene_symbol":r.gene_symbol,
                "core_target_chembl_id":r.core_target_chembl_id,
                "core_accession_state":"ERROR",
                "core_accessions":acc,
                "error":terr,
            })
            continue
        for h in hits:
            seen[h["target_chembl_id"]]=h

    for h in seen.values():
        target_rows.append({
            **r.to_dict(),
            **h,
        })
        mm,mstate,merr=mechanisms_for_target(h["target_chembl_id"])
        if mstate=="ERROR":
            continue
        for m in mm:
            mi=molecule_info(m.get("molecule_chembl_id"))
            mech_rows.append({
                "branch":r.branch,
                "target_symbol":r.gene_symbol,
                "matched_accession":h["matched_accession"],
                "resolved_target_chembl_id":h["target_chembl_id"],
                "resolved_target_name":h["pref_name"],
                "resolved_target_type":h["target_type"],
                "molecule_chembl_id":m.get("molecule_chembl_id"),
                "drug":mi.get("pref_name"),
                "parent_chembl_id":mi.get("parent_chembl_id"),
                "max_phase":mi.get("max_phase"),
                "first_approval":mi.get("first_approval"),
                "mechanism_of_action":m.get("mechanism_of_action"),
                "action_type":m.get("action_type"),
                "molecule_query_error":mi.get("query_error"),
            })

CORE_RESOLUTION=pd.DataFrame(resolution_rows)
TARGETS=pd.DataFrame(target_rows).drop_duplicates()
MECH=pd.DataFrame(mech_rows)

if len(MECH):
    MECH["approved_phase4"] = pd.to_numeric(MECH["max_phase"], errors="coerce").eq(4)
    MECH["canonical_drug_key"] = MECH["parent_chembl_id"].fillna(MECH["molecule_chembl_id"])
else:
    MECH=pd.DataFrame(columns=[
        "branch","target_symbol","matched_accession",
        "resolved_target_chembl_id","resolved_target_name","resolved_target_type",
        "molecule_chembl_id","drug","parent_chembl_id","max_phase",
        "first_approval","mechanism_of_action","action_type",
        "molecule_query_error","approved_phase4","canonical_drug_key"
    ])

CORE_RESOLUTION.to_csv(OUT/"NB08R2_CORE_ACCESSION_RESOLUTION.csv",index=False)
TARGETS.to_csv(OUT/"NB08R2_CHEMBL_TARGET_RESOLUTION.csv",index=False)
MECH.to_csv(OUT/"NB08R2_CHEMBL_MECHANISMS_ALL.csv",index=False)

display(CORE_RESOLUTION)
display(TARGETS[[
    "gene_symbol","matched_accession","target_chembl_id","pref_name","target_type"
]].sort_values(["gene_symbol","target_type","pref_name"]))


## 6. Explicit source-state coverage table

`chembl_activity_state` defaults to `NOT_RUN`. It is never encoded as a numeric zero.  
If a future bounded activity analysis is added, it must write a separate artifact and replace that state explicitly.


In [ ]:
coverage=[]
for _,r in REGISTRY.iterrows():
    t=TARGETS[TARGETS["gene_symbol"].eq(r.gene_symbol)] if len(TARGETS) else pd.DataFrame()
    m=MECH[(MECH["target_symbol"].eq(r.gene_symbol)) & (MECH["approved_phase4"])] if len(MECH) else pd.DataFrame()
    d=DC_AUDIT[DC_AUDIT["target_symbol"].eq(r.gene_symbol)].iloc[0]

    rr=CORE_RESOLUTION[CORE_RESOLUTION["gene_symbol"].eq(r.gene_symbol)]
    if (rr["core_accession_state"]=="ERROR").any():
        chem_state="ERROR"
    elif len(t)==0:
        chem_state="NO_TARGET_RESOLVED"
    else:
        chem_state="HIT" if len(m) else "NO_HIT"

    coverage.append({
        "branch":r.branch,
        "target_symbol":r.gene_symbol,
        "core_target_chembl_id":r.core_target_chembl_id,
        "core_accessions":";".join(sorted(set(
            ";".join(rr["core_accessions"].dropna().astype(str)).split(";")
        ) - {""})),
        "desired_action":r.desired_action,
        "role":r.role,
        "chembl_target_representations_n":len(t),
        "chembl_mechanism_state":chem_state,
        "chembl_approved_mechanism_unique_parent_drugs":
            int(m["canonical_drug_key"].nunique()) if len(m) else 0,
        "chembl_activity_state":"NOT_RUN",
        "chembl_activity_unique_drugs":pd.NA,
        "drugcentral_state":d.drugcentral_query_state,
        "drugcentral_unique_approved_structures":int(d.drugcentral_unique_approved_structures),
    })

COV=pd.DataFrame(coverage)
COV.to_csv(OUT/"NB08R2_COVERAGE_STATES.csv",index=False)
display(COV)


## 7. Required precision and positive-control assertions

v1.0.2 is rejected if:

- the frozen DrugCentral snapshot fails to recover known PSMB5 proteasome inhibitors;
- a resolved ChEMBL target does not actually contain the matched UniProt accession;
- known fuzzy-search false positives from v1.0.1 leak back into the target set;
- the frozen core PSMB5 target disappears.


In [ ]:
# DrugCentral frozen-snapshot positive controls.
psmb5 = dc[(dc["approved_snapshot"]) & (dc["GENE"].astype(str).eq("PSMB5"))]
names=set(psmb5["DRUG_NAME"].astype(str).str.lower())
assert "bortezomib" in names, "Positive-control failure: bortezomib not recovered for PSMB5"
assert "ixazomib" in names, "Positive-control failure: ixazomib not recovered for PSMB5"

# Every ChEMBL resolved target must contain the exact core UniProt accession.
assert len(TARGETS)>0, "No exact component-aware ChEMBL targets resolved"
for _,x in TARGETS.iterrows():
    assert x["matched_accession"] in set(str(x["all_component_accessions"]).split(";"))

# Precision sanity check: the v1.0.1 fuzzy-search artifacts must disappear.
for bad in ["Steroidogenic factor 1","Solute carrier family 26 member 9","Solute carrier family 26 member 6"]:
    assert not TARGETS["pref_name"].astype(str).eq(bad).any(), f"Fuzzy target leaked into exact mapping: {bad}"

# Proteasome branch should contain at least one actual proteasome complex if ChEMBL models it
# with the PSMB5 accession. The single-protein PSMB5 target must always be present.
prot = TARGETS[TARGETS["gene_symbol"].eq("PSMB5")]
assert prot["target_chembl_id"].astype(str).eq("CHEMBL4662").any(), "Core PSMB5 target missing"

print("PSMB5 exact component-bearing target representations:")
display(prot[["matched_accession","target_chembl_id","pref_name","target_type"]].drop_duplicates())

# ChEMBL mechanism positive-control: if bortezomib is annotated against any PSMB5-containing
# representation, show it. Do not fail solely on nomenclature if ChEMBL's mechanism record is absent.
pm = MECH[(MECH["target_symbol"].eq("PSMB5")) & (MECH["approved_phase4"])]
bz = pm[pm["drug"].astype(str).str.lower().eq("bortezomib")]
print("ChEMBL bortezomib mechanism rows on PSMB5-containing targets:", len(bz))
if len(bz):
    display(bz[[
        "drug","resolved_target_chembl_id","resolved_target_name",
        "resolved_target_type","mechanism_of_action","action_type"
    ]].drop_duplicates())

print("NB08R2 v1.0.2 precision audit gates: PASS")


## 8. Interpretation contract

This notebook measures **mechanism-annotation coverage on exact protein-component targets**.

Allowed statements:
- a ChEMBL mechanism query was completed over targets that actually contain the protein's UniProt accession;
- a source returned `HIT`, `NO_HIT`, `ERROR`, or `NOT_RUN`;
- approved mechanism records exist or do not exist for those exact representations.

Not allowed:
- equating annotation density with therapeutic credibility;
- treating `NOT_RUN` activity as zero;
- treating every mechanism hit as directionally useful;
- treating a target-complex hit as evidence of rescue;
- issuing a final numerical drug ranking.

Directionality, exposure, chronic feasibility, and the F0 abundance/function gate remain separate layers.


In [ ]:
handoff={
    "schema":"MVA-NB08R2-COVERAGE-AUDIT-1.0.2",
    "ranking_used_for_selection":False,
    "purpose":"exact_component_aware_coverage_and_falsification",
    "states":["HIT","NO_HIT","NOT_RUN","ERROR","NO_TARGET_RESOLVED"],
    "activity_channel_default":"NOT_RUN",
    "drugcentral_approval_filter_applied":True,
    "drugcentral_dedup_key":"STRUCT_ID",
    "chembl_target_mapping":"core_target -> exact UniProt accession -> target_components__accession",
    "free_text_target_search_used":False,
    "complex_target_resolution":True,
    "proteasome_positive_control_required":True,
    "snapshot_mode":SNAPSHOT_MODE,
    "snapshot_directory":str(SNAP),
    "outputs":[
        "NB08R2_CORE_ACCESSION_RESOLUTION.csv",
        "NB08R2_CHEMBL_TARGET_RESOLUTION.csv",
        "NB08R2_CHEMBL_MECHANISMS_ALL.csv",
        "NB08R2_COVERAGE_STATES.csv",
        "NB08R2_HANDOFF.json"
    ]
}
(OUT/"NB08R2_HANDOFF.json").write_text(json.dumps(handoff,indent=2))

# Persist all final outputs to Drive when the strengthening package was found there.
if PERSIST is not None:
    for fn in handoff["outputs"]:
        src=OUT/fn
        if src.exists():
            shutil.copy2(src, PERSIST/fn)
    print("Persisted outputs to:", PERSIST)

print(json.dumps(handoff,indent=2))
